**Create Table on Big Query**

In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# GOOGLE BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

# =========================================
# CONFIGURATION
# =========================================

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "ICD_Codes"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# =========================================
# INITIALIZE BIGQUERY CLIENT
# =========================================

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# TABLE SCHEMA
# =========================================

schema = [

    # Primary Key
    bigquery.SchemaField(
        "ICD_Code",
        "STRING",
        mode="REQUIRED"
    ),

    bigquery.SchemaField(
        "ICD_Code_Description",
        "STRING"
    ),

    bigquery.SchemaField(
        "ICD_Main_Disease_Code",
        "STRING"
    ),

    bigquery.SchemaField(
        "ICD_Main_Disease_Description",
        "STRING"
    ),

    bigquery.SchemaField(
        "ICD_Disease_Category_Sub_Chapter_Code",
        "STRING"
    ),

    bigquery.SchemaField(
        "ICD_Disease_Category_Sub_Chapter_Description",
        "STRING"
    ),

    bigquery.SchemaField(
        "ICD_Disease_Category_Sub_Chapter_Description_AR",
        "STRING"
    ),

    bigquery.SchemaField(
        "ICD_Disease_Category_Chapter_Code",
        "STRING"
    ),

    bigquery.SchemaField(
        "ICD_Disease_Category_Chapter_Description",
        "STRING"
    ),

    bigquery.SchemaField(
        "ICD_Chronic",
        "BOOLEAN"
    )
]

# =========================================
# CREATE TABLE OBJECT
# =========================================

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

# =========================================
# CLUSTERING
# =========================================

table.clustering_fields = [
    "ICD_Code",
    "ICD_Main_Disease_Code"
]

# =========================================
# CREATE TABLE
# =========================================

table = client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("Table created successfully")
print(TABLE_REF)
print("=================================")

Table created successfully
depihealthnux.Depihealthnux_Main.ICD_Codes


**Extracting Data from WHO XML File & Adding Arabic Translation**

In [ ]:
import pandas as pd
from lxml import etree
import re

# =========================================
# CONFIGURATION
# =========================================

XML_FILE = "../Resources/icd10c-tabular-April-1-2026.xml"
AR_TRANSLATION_FILE = "../Resources/ICD10_Description_AR.xlsx"

# =========================================
# ICD EXTRACTION
# =========================================

rows = []

def walk_diag(diag, context):
    code = diag.findtext("name")
    desc = diag.findtext("desc")

    children = diag.findall("diag")

    if not children:
        rows.append({
            "ICD Code": code,
            "ICD Code Description": desc,
            "ICD Main Disease Code": context.get("main_code"),
            "ICD Main Disease Description": context.get("main_desc"),
            "ICD Disease Category Sub Chapter Code": context.get("section_code"),
            "ICD Disease Category Sub Chapter Description": context.get("section_desc"),
            "ICD Disease Category Chapter Code": context.get("chapter_code"),
            "ICD Disease Category Chapter Description": context.get("chapter_desc"),
        })
    else:
        for child in children:
            walk_diag(
                child,
                {
                    **context,
                    "main_code": code,
                    "main_desc": desc,
                }
            )

def parse_xml():
    tree = etree.parse(XML_FILE)
    root = tree.getroot()

    for chapter in root.findall("chapter"):
        chapter_code = chapter.findtext("name")
        chapter_desc = chapter.findtext("desc")

        for section in chapter.findall("section"):
            section_code = section.get("id")
            section_desc = section.findtext("desc")

            for diag in section.findall("diag"):
                walk_diag(
                    diag,
                    {
                        "chapter_code": chapter_code,
                        "chapter_desc": chapter_desc,
                        "section_code": section_code,
                        "section_desc": section_desc,
                        "main_code": diag.findtext("name"),
                        "main_desc": diag.findtext("desc"),
                    }
                )

# =========================================
# EXTRACT XML
# =========================================

parse_xml()

df = pd.DataFrame(rows)

# =========================================
# CLEAN ICD TEXT COLUMNS
# =========================================

paren_pattern = re.compile(r"\s*\([^)]*\)")

icd_cols = [c for c in df.columns if "ICD" in c]

for c in icd_cols:
    df[c] = (
        df[c]
        .astype(str)
        .str.replace(paren_pattern, "", regex=True)
        .str.strip()
        .replace({"None": ""})
    )

# =========================================
# LOAD ARABIC TRANSLATIONS
# =========================================

ar_df = pd.read_excel(AR_TRANSLATION_FILE)

# Clean lookup column
ar_df["ICD_Disease_Category_Sub_Chapter_Description"] = (
    ar_df["ICD_Disease_Category_Sub_Chapter_Description"]
    .astype(str)
    .str.strip()
)

# Clean source column
df["ICD Disease Category Sub Chapter Description"] = (
    df["ICD Disease Category Sub Chapter Description"]
    .astype(str)
    .str.strip()
)

# =========================================
# MERGE Arabic translation the df
# =========================================

df = df.merge(
    ar_df[
        [
            "ICD_Disease_Category_Sub_Chapter_Description",
            "Arabic_Translation"
        ]
    ],
    left_on="ICD Disease Category Sub Chapter Description",
    right_on="ICD_Disease_Category_Sub_Chapter_Description",
    how="left"
)

# Remove duplicate key column from lookup file
df.drop(
    columns=["ICD_Disease_Category_Sub_Chapter_Description"],
    inplace=True
)

# Insert Arabic column beside English column
insert_pos = (
    df.columns.get_loc(
        "ICD Disease Category Sub Chapter Description"
    ) + 1
)

arabic_values = df.pop("Arabic_Translation")
df.insert(
    insert_pos,
    "ICD Disease Category Sub Chapter Description AR",
    arabic_values
)

df["ICD Disease Category Sub Chapter Description AR"] = (
    df["ICD Disease Category Sub Chapter Description AR"]
    .fillna("")
)

# =========================================
# FINAL COLUMN NAMES
# =========================================

df.columns = [col.replace(" ", "_") for col in df.columns]


print(f"\nTotal ICD codes: {len(df):,}")

matched = (
    df["ICD_Disease_Category_Sub_Chapter_Description_AR"]
    .ne("")
    .sum()
)

print(f"Arabic translations matched: {matched:,}")
with pd.option_context(
    "display.max_columns", None,
    "display.width", 300,
    "display.max_colwidth", 50,
    "display.unicode.east_asian_width", True
):
    print("\nFirst 10 Rows:")
    print(df.head(10).to_string(index=False))


Total ICD codes: 36,343
Arabic translations matched: 36,343

First 10 Rows:
ICD_Code                               ICD_Code_Description ICD_Main_Disease_Code ICD_Main_Disease_Description ICD_Disease_Category_Sub_Chapter_Code ICD_Disease_Category_Sub_Chapter_Description ICD_Disease_Category_Sub_Chapter_Description_AR ICD_Disease_Category_Chapter_Code  ICD_Disease_Category_Chapter_Description
   A00.0 Cholera due to Vibrio cholerae 01, biovar cholerae                   A00                      Cholera                               A00-A09               Intestinal infectious diseases                         الأمراض المعوية المعدية                                 1 Certain infectious and parasitic diseases
   A00.1    Cholera due to Vibrio cholerae 01, biovar eltor                   A00                      Cholera                               A00-A09               Intestinal infectious diseases                         الأمراض المعوية المعدية                                 1 Certain inf

In [ ]:
# =====================================================
# CHRONIC CONDITION FLAGGING
# =====================================================

import pandas as pd
import re

CHRONIC_CONDITIONS = {

    # =====================
    # ENDOCRINE
    # =====================
    "Diabetes": [("E08", "E13")],
    "Dyslipidemia": [("E78", "E78")],
    "Hypothyroidism": [("E03", "E03")],
    "Hyperthyroidism": [("E05", "E05")],
    "Vitamin D Deficiency": [("E55", "E55")],
    "Obesity": [("E66", "E66")],
    "Polycystic Ovary Syndrome": [("E28", "E28")],

    # =====================
    # CARDIOVASCULAR
    # =====================
    "Hypertension": [("I10", "I15")],
    "Ischemic Heart Disease": [("I20", "I25")],
    "Heart Failure": [("I50", "I50")],
    "Atrial Fibrillation": [("I48", "I48")],
    "Peripheral Vascular Disease": [("I70", "I79")],
    "Cerebrovascular Disease": [("I60", "I69")],
    "Pulmonary Hypertension": [("I27", "I27")],
    "Cardiomyopathy": [("I42", "I43")],
    "Rheumatic Heart Disease": [("I05", "I09")],
    "Chronic Venous Insufficiency": [("I87", "I87")],

    # =====================
    # RESPIRATORY
    # =====================
    "COPD": [("J44", "J44")],
    "Asthma": [("J45", "J45")],
    "Bronchiectasis": [("J47", "J47")],
    "Chronic Sinusitis": [("J32", "J32")],
    "Sleep Apnea": [("G47", "G47")],

    # =====================
    # RENAL
    # =====================
    "Chronic Kidney Disease": [("N18", "N18")],
    "Nephrotic Syndrome": [("N04", "N04")],

    # =====================
    # NEUROLOGY
    # =====================
    "Epilepsy": [("G40", "G41")],
    "Parkinson Disease": [("G20", "G20")],
    "Alzheimer Disease": [("G30", "G30")],
    "Multiple Sclerosis": [("G35", "G35")],
    "Migraine": [("G43", "G43")],
    "Myasthenia Gravis": [("G70", "G70")],
    "Motor Neuron Disease": [("G12", "G12")],
    "Huntington Disease": [("G10", "G10")],

    # =====================
    # MENTAL HEALTH
    # =====================
    "Depression": [("F32", "F33")],
    "Bipolar Disorder": [("F31", "F31")],
    "Schizophrenia": [("F20", "F29")],
    "Anxiety Disorder": [("F40", "F41")],

    # =====================
    # RHEUMATOLOGY
    # =====================
    "Rheumatoid Arthritis": [("M05", "M06")],
    "Osteoarthritis": [("M15", "M19")],
    "Osteoporosis": [("M80", "M81")],
    "Systemic Lupus Erythematosus": [("M32", "M32")],
    "Systemic Sclerosis": [("M34", "M34")],
    "Sjogren Syndrome": [("M35", "M35")],
    "Ankylosing Spondylitis": [("M45", "M45")],
    "Fibromyalgia": [("M79", "M79")],
    "Gout": [("M10", "M10")],
    "Vasculitis": [("M30", "M31")],
    "Chronic Osteomyelitis": [("M86", "M86")],

    # =====================
    # GASTRO
    # =====================
    "GERD": [("K21", "K21")],
    "Crohn Disease": [("K50", "K50")],
    "Ulcerative Colitis": [("K51", "K51")],
    "Inflammatory Bowel Disease": [("K50", "K51")],
    "Irritable Bowel Syndrome": [("K58", "K58")],
    "Chronic Liver Disease": [("K70", "K77")],
    "Cirrhosis": [("K74", "K74")],
    "Chronic Pancreatitis": [("K86", "K86")],

    # =====================
    # INFECTIOUS
    # =====================
    "HIV/AIDS": [("B20", "B24")],
    "Chronic Viral Hepatitis": [("B18", "B18")],
    "Sarcoidosis": [("D86", "D86")],

    # =====================
    # HEMATOLOGY
    # =====================
    "Iron Deficiency Anemia": [("D50", "D50")],
    "Vitamin B12 Deficiency Anemia": [("D51", "D51")],
    "Folate Deficiency Anemia": [("D52", "D52")],
    "Other Nutritional Anemia": [("D53", "D53")],
    "Hemolytic Anemia": [("D55", "D59")],
    "Aplastic Anemia": [("D60", "D64")],
    "Thalassemia": [("D56", "D56")],
    "Sickle Cell Disease": [("D57", "D57")],
    "Hemophilia": [("D66", "D68")],

    # =====================
    # DERMATOLOGY
    # =====================
    "Psoriasis": [("L40", "L40")],

    # =====================
    # EYE
    # =====================
    "Glaucoma": [("H40", "H42")],
    "Cataract": [("H25", "H28")],

    # =====================
    # GU
    # =====================
    "Benign Prostatic Hyperplasia": [("N40", "N40")],
    "Endometriosis": [("N80", "N80")],
    "Chronic Cystitis": [("N30", "N30")]
}


def icd_prefix(icd):

    if pd.isna(icd):
        return ""

    icd = str(icd).upper().strip()

    if "." in icd:
        icd = icd.split(".")[0]

    return icd[:3]


def classify_chronic(icd_code,
                     description,
                     subchapter_desc):

    description = str(description).lower()
    prefix = icd_prefix(icd_code)

    # Cancer
    if prefix and prefix[0] == "C":
        return pd.Series([
            "Yes",
            "Cancer",
            "ICD Mapping"
        ])

    # D00-D49 Neoplasms
    if prefix.startswith("D"):
        try:
            d_num = int(prefix[1:])
            if d_num <= 49:
                return pd.Series([
                    "Yes",
                    "Cancer",
                    "ICD Mapping"
                ])
        except:
            pass

    # Z status codes
    if prefix in ["Z94", "Z79"]:
        return pd.Series([
            "Yes",
            "Long-Term Medical Status",
            "ICD Mapping"
        ])

    # Standard mapping
    try:

        letter = prefix[0]
        number = int(prefix[1:])

        for condition, ranges in CHRONIC_CONDITIONS.items():

            for start, end in ranges:

                if (
                    start[0] == letter
                    and end[0] == letter
                ):

                    start_num = int(start[1:])
                    end_num = int(end[1:])

                    if start_num <= number <= end_num:

                        return pd.Series([
                            "Yes",
                            condition,
                            "ICD Mapping"
                        ])

    except:
        pass

    # Chronic keyword fallback
    if re.search(r"\bchronic\b", description):

        return pd.Series([
            "Yes",
            str(subchapter_desc),
            "Description Keyword"
        ])

    return pd.Series([
        "No",
        "",
        ""
    ])


df[
    [
        "Is_Chronic",
        "Chronic_Condition",
        "Chronic_Source"
    ]
] = df.apply(
    lambda row: classify_chronic(
        row["ICD_Code"],
        row["ICD_Code_Description"],
        row["ICD_Disease_Category_Sub_Chapter_Description"]
    ),
    axis=1
)

print(f"Total ICD Codes: {len(df):,}")
print(f"Chronic Codes: {(df['Is_Chronic'] == 'Yes').sum():,}")

print("\nTop 50 Chronic Conditions\n")
print(
    df[df["Is_Chronic"] == "Yes"]
    .groupby("Chronic_Condition")
    .size()
    .sort_values(ascending=False)
    .head(10)
)


Saved: ICD10_FULL_DENORMALIZED_WITH_CHRONIC.xlsx
Total ICD Codes: 36,343
Chronic Codes: 5,804

Top 50 Chronic Conditions

Chronic_Condition
Cancer                                                                            1705
Cerebrovascular Disease                                                            410
Peripheral Vascular Disease                                                        344
Rheumatoid Arthritis                                                               312
Other disorders of the skin and subcutaneous tissue                                304
Diabetes                                                                           237
Chronic Osteomyelitis                                                              179
Inflammatory polyarthropathies                                                     145
Endometriosis                                                                      138
Gout                                                                        

In [14]:
# =========================================
# ADD MISSING COLUMNS IF NOT EXISTS
# =========================================

table = client.get_table(TABLE_REF)

existing_columns = {field.name for field in table.schema}

new_columns = []

if "ICD_Chronic" not in existing_columns:
    new_columns.append(
        bigquery.SchemaField(
            "ICD_Chronic",
            "BOOLEAN",
            mode="NULLABLE"
        )
    )

if "ICD_Chronic_Condition" not in existing_columns:
    new_columns.append(
        bigquery.SchemaField(
            "ICD_Chronic_Condition",
            "STRING",
            mode="NULLABLE"
        )
    )

if "ICD_Chronic_Source" not in existing_columns:
    new_columns.append(
        bigquery.SchemaField(
            "ICD_Chronic_Source",
            "STRING",
            mode="NULLABLE"
        )
    )

if new_columns:

    updated_schema = list(table.schema) + new_columns

    table.schema = updated_schema

    table = client.update_table(
        table,
        ["schema"]
    )

    print(
        f"Added {len(new_columns)} new columns."
    )

else:

    print(
        "All chronic columns already exist."
    )

# =========================================
# PREPARE DATAFRAME
# =========================================

bq_df = df.copy()

# Convert Yes/No to Boolean
bq_df["ICD_Chronic"] = (
    bq_df["Is_Chronic"]
    .fillna("No")
    .eq("Yes")
)

bq_df["ICD_Chronic_Condition"] = (
    bq_df["Chronic_Condition"]
    .fillna("")
)

bq_df["ICD_Chronic_Source"] = (
    bq_df["Chronic_Source"]
    .fillna("")
)

# Keep only columns existing in BQ schema
bq_df = bq_df[
    [
        "ICD_Code",
        "ICD_Code_Description",
        "ICD_Main_Disease_Code",
        "ICD_Main_Disease_Description",
        "ICD_Disease_Category_Sub_Chapter_Code",
        "ICD_Disease_Category_Sub_Chapter_Description",
        "ICD_Disease_Category_Sub_Chapter_Description_AR",
        "ICD_Disease_Category_Chapter_Code",
        "ICD_Disease_Category_Chapter_Description",
        "ICD_Chronic",
        "ICD_Chronic_Condition",
        "ICD_Chronic_Source"
    ]
]

# =========================================
# LOAD TO BIGQUERY
# =========================================

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

load_job = client.load_table_from_dataframe(
    bq_df,
    TABLE_REF,
    job_config=job_config
)

load_job.result()

print(
    f"Loaded {len(bq_df):,} rows into {TABLE_REF}"
)

# =========================================
# VALIDATION
# =========================================

table = client.get_table(TABLE_REF)

print(
    f"Rows in table: {table.num_rows:,}"
)

All chronic columns already exist.
Loaded 36,343 rows into depihealthnux.Depihealthnux_Main.ICD_Codes
Rows in table: 36,343
